## Imports

In [1]:
from qiskit.circuit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService, SamplerV2 as Sampler

from schwinger_hamiltonian import *

In [2]:
service = QiskitRuntimeService()
backend = service.backend(name='ibm_rensselaer')

qiskit_runtime_service._resolve_cloud_instances:WARNING:2025-08-06 10:28:53,952: Default instance not set. Searching all available instances.


In [3]:
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

In [4]:
sampler = Sampler(mode=backend)
sampler.options.default_shots = 8192

estimator = Estimator(mode=backend)
estimator.options.default_shots = 8192

## Central Dogma of Adaptive Trotterization
The most important aspect that makes adaptive trotterization a viable algorithm for simulating complex systems is maximizing the time spent per step whilst keeping error rates low.

## Defining Step Limit

In [ ]:
def sequential_search(dt: float) -> float:
  N = 1
  largest_step = 0
  step_reduction = 0.01
  flag = True
  i = 0
  longest_loop = 100

  while(flag and i >= longest_loop):
    if energy_function(dt) < 0:
      flag = False
    else:
      dt -= step_reduction
      i += 1

  return dt

In [ ]:
def bisectional_search(dt: float) -> float:
  M = 1
  R = 1
  R_max = 100
  while(R < R_max and flag):
    

  return dt

In [14]:
test_circ = QuantumCircuit(3)
test_results = []

for i in range(2):
  test_circ.h(0)
  test_circ.h(1)
  test_circ.h(2)
  test_circ.cx(0, 1)
  test_circ.cx(1, 2)
  test_circ.rz(np.pi / 12, 2)
  test_circ.cx(1, 2)
  test_circ.cx(0, 1)
  test_circ.h(0)
  test_circ.h(1)
  test_circ.h(2)
  test_circ.measure_all()

  isa_test_circ = pm.run(test_circ)
  test_hamiltonian = SparsePauliOp.from_list([("ZII", 1), ("IZI", 1), ("IIZ", 1)])
  test_hamiltonian = test_hamiltonian.apply_layout(isa_test_circ.layout)

  test_job = estimator.run([(isa_test_circ, test_hamiltonian)])
  test_result = test_job.result()
  test_results.append(test_result)

In [19]:
test_results[0].__dict__

{'_pub_results': [PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>)), metadata={'shots': 8192, 'target_precision': 0.011048543456039804, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})],
 '_metadata': {'dynamical_decoupling': {'enable': False,
   'sequence_type': 'XX',
   'extra_slack_distribution': 'middle',
   'scheduling_method': 'alap'},
  'twirling': {'enable_gates': False,
   'enable_measure': True,
   'num_randomizations': 'auto',
   'shots_per_randomization': 'auto',
   'interleave_randomizations': True,
   'strategy': 'active-accum'},
  'resilience': {'measure_mitigation': True,
   'zne_mitigation': False,
   'pec_mitigation': False},
  'version': 2}}

In [26]:
test_results[0][0].data.evs

array(2.73060241)

In [28]:
for i in range(2):
  print(test_results[i][0].data.evs)
  print(test_results[i][0].data.stds)

2.730602406561675
0.00967598007866638
2.54301856749667
0.011178231253036582


This is a function that steps forward the trotterization circuit by one time step

In [5]:
def step_forward_ising(circ: QuantumCircuit) -> None:
  circ.h(0)
  circ.h(1)
  circ.h(2)
  circ.cx(0, 1)
  circ.cx(1, 2)
  circ.rz(np.pi / 12, 2)
  circ.cx(1, 2)
  circ.cx(0, 1)
  circ.h(0)
  circ.h(1)
  circ.h(2)

## Ising Model Trial
The Ising Model is one of the easiest systems to simulate, as it is just a 1D chain of spins.

In [ ]:
# Variable initialization
perm_circ = QuantumCircuit(3)
temp_circ = QuantumCircuit(3)
step = 1
ising_hamiltonian = SparsePauliOp.from_list([("ZII", 1), ("IZI", 1), ("IIZ", 1)])
results = []

# The first time step
step_forward_ising(temp_circ)
isa_temp_circ = pm.run(temp_circ)
isa_hamiltonian = ising_hamiltonian.apply_layout(isa_temp_circ.layout)
job = estimator.run([(isa_temp_circ, isa_hamiltonian)])
result = job.result()
results.append(result)
perm_circ = temp_circ

for step in range(20):
  flag = True
  temp_steps = 0
  while(flag or temp_steps < 20):
    step_forward_ising(temp_circ)
    temp_steps += 1
    temp_circ.measure_all()

    isa_temp_circ = pm.run(temp_circ)
    isa_hamiltonian = ising_hamiltonian.apply_layout(isa_temp_circ.layout)

    temp_job = estimator.run([(isa_temp_circ, isa_hamiltonian)])
    temp_result = temp_job.result()

    if (abs(temp_result[0].data.evs - results[len(results) - 1][0].data.evs) / temp_steps > 0.2 or abs(temp_result[0].data.evs - results[len(results) - 1][0].data.evs) / temp_steps < 0.18):
      flag = False
      step += temp_steps
      break
    
    perm_result = temp_result
    results.append(perm_result)
    temp_circ.clear()
    for i in range(temp_steps):
      step_forward_ising(temp_circ)

  perm_circ = temp_circ
  temp_circ.clear()


KeyboardInterrupt: 

## N=4 Circuit Construction

In [9]:
# Model parameters
N = 4
m = 1
a = 1
g = 0
J = 0.5*g*g*a

# Backend parameters
t_start = 0
t_max = 3
dt = 0.2
shots = 10000
time_step_max = int(t_max/dt)
#time_step_list = np.linspace(t_start, t_max, (int((t_max - t_start)/dt))+1)

# quantum and measurement circuits
qr = QuantumRegister(N)
cr = ClassicalRegister(N)
qc = QuantumCircuit(qr,cr)
qc_meas = QuantumCircuit(qr,cr)
qc_meas.measure(qr,cr)

# initial state prep
for i in range(int(N/2)):
  qc.x(qr[2*i+1])

time_list = []
cc_list_n4_g0 = []
elec_list_n4_g0 = []
circuit_depth_list = []
evolution_steps = []

for time_step in range(time_step_max+1):
    print("Time = ", dt*time_step)

    # H_{XX+YY} kinetic term
    for i in range(N-1):
        qc.cx(qr[i], qr[i+1])
        qc.h(qr[i])
        qc.cx(qr[i], qr[i+1])

        qc.rz((0.5/a)*dt, qr[i])
        qc.rz(-(0.5/a)*dt, qr[i+1])

        qc.cx(qr[i], qr[i+1])
        qc.h(qr[i])
        qc.cx(qr[i], qr[i+1])

    # H_Z mass term
    for i in range(N):
        qc.rz(m*((-1)**(i+1))*dt, qr[i])

    # H_ZZ electric field term
    for i in range(1, N-1):
        for k in range(0, i):
            for l in range(k+1, i+1):
                qc.rzz(J*dt, qr[k], qr[l])

    # H_Z electric field term
    for n in range(N-1):
        qc.rz(-0.5*J*dt*(N-(n+1)-0.5*(-1+(-1)**(n+1))), qr[n])

    qc_total = qc.compose(qc_meas)
    evolution_steps.append(qc_total)


Time =  0.0
Time =  0.2
Time =  0.4
Time =  0.6000000000000001
Time =  0.8
Time =  1.0
Time =  1.2000000000000002
Time =  1.4000000000000001
Time =  1.6
Time =  1.8
Time =  2.0
Time =  2.2
Time =  2.4000000000000004
Time =  2.6
Time =  2.8000000000000003
Time =  3.0
